# 🚀 Google Colab 클라우드 런타임 환경 & Pandas 설정 가이드

이 주피터 노트북은 **구글 코랩(Google Colab)** 클라우드 런타임에서 **Pandas**를 비롯한 데이터 분석 라이브러리를 즉시 실행할 수 있도록 구성된 템플릿입니다.

### 💡 코랩 클라우드 환경의 특징
- **사전 설치된 라이브러리**: Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn 등이 이미 설치되어 있어 별도 로컬 설치가 필요 없습니다.
- **클라우드 컴퓨팅 자원**: 고성능 CPU 및 무료 T4 GPU를 바로 활용할 수 있습니다.
- **구글 드라이브 연동**: 대용량 데이터셋과 분석 결과 파일을 클라우드에 쉽게 보관할 수 있습니다.

## 1. 클라우드 런타임 사양 및 가속기(GPU/CPU) 확인
현재 연결된 구글 클라우드 가상머신의 사양과 가속기 상태를 확인합니다.

In [ ]:
# 런타임 하드웨어 및 OS 사양 확인
import sys
import platform
import os

print(f"📌 Python 버전: {sys.version.split()[0]}")
print(f"📌 실행 OS: {platform.platform()}")

# GPU 활성화 확인 (상단 메뉴: 런타임 > 런타임 유형 변경 > T4 GPU 선택 가능)
try:
    import torch
    if torch.cuda.is_available():
        print(f"🔥 GPU 활성화됨: {torch.cuda.get_device_name(0)}")
    else:
        print("⚡ 현재 CPU 런타임 사용 중 (데이터 분석 및 Pandas 작업용으로 충분합니다)")
except ImportError:
    print("⚡ 기본 CPU 환경")

## 2. 필수 라이브러리(Pandas 등) 확인 및 추가 패키지 설치
Google Colab에는 Pandas가 기본 설치되어 있습니다. 필요한 경우 버전을 확인하거나 추가 라이브러리를 설치합니다.

In [ ]:
# 핵심 데이터 분석 라이브러리 로드
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"✅ Pandas 버전     : {pd.__version__}")
print(f"✅ NumPy 버전      : {np.__version__}")
print(f"✅ Matplotlib 버전 : {plt.matplotlib.__version__}")
print(f"✅ Seaborn 버전    : {sns.__version__}")

# 추가 라이브러리가 필요할 경우 아래처럼 '!pip install' 명령어로 설치할 수 있습니다.
# !pip install -q openpyxl plotly scikit-learn

## 3. 구글 드라이브(Google Drive) 연동
데이터 파일(.csv, .xlsx 등)을 읽고 쓰기 위해 구글 드라이브를 마운트합니다.

In [ ]:
# 구글 드라이브 마운트
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("🎉 구글 드라이브 마운트 성공! 경로: /content/drive/MyDrive/")
except Exception as e:
    print(f"ℹ️ 로컬 환경이거나 마운트 건너뜀: {e}")

## 4. Pandas 데이터 처리 및 분석 실습 예제
클라우드 런타임에서 실제 데이터를 생성하고 가공, 그룹 분석하는 예제입니다.

In [ ]:
# 샘플 전자상거래 데이터셋 생성
raw_data = {
    '주문번호': ['ORD-001', 'ORD-002', 'ORD-003', 'ORD-004', 'ORD-005', 'ORD-006', 'ORD-007', 'ORD-008'],
    '고객명': ['김철수', '이영희', '박민호', '정수진', '최지훈', '한예슬', '강동원', '윤서아'],
    '카테고리': ['전자제품', '패션의류', '신선식품', '전자제품', '도서음반', '패션의류', '신선식품', '전자제품'],
    '수량': [1, 3, 10, 2, 5, 2, 4, 1],
    '단가': [1250000, 48000, 8500, 320000, 19500, 89000, 15000, 890000],
    '지역': ['서울', '경기', '부산', '서울', '대전', '경기', '서울', '대구']
}

df = pd.DataFrame(raw_data)

# 파생 변수 생성: 총금액 = 수량 * 단가
df['총금액'] = df['수량'] * df['단가']

print("📋 [데이터프레임 미리보기]")
display(df)

print("\n📊 [카테고리별 매출 요약 집계]")
category_summary = df.groupby('카테고리').agg(
    총매출=('총금액', 'sum'),
    평균단가=('단가', 'mean'),
    주문건수=('주문번호', 'count')
).reset_index().sort_values(by='총매출', ascending=False)

display(category_summary)

## 5. 데이터 시각화
집계된 결과를 Matplotlib/Seaborn으로 시각화합니다.

In [ ]:
# 카테고리별 총매출 시각화 바 차트
plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary, x='카테고리', y='총매출', palette='Blues_r')
plt.title('Sales by Category (Google Colab Cloud)', fontsize=14, pad=15)
plt.xlabel('Category', fontsize=11)
plt.ylabel('Total Sales (KRW)', fontsize=11)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: f"{int(x):,}"))
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 6. 분석 결과 내보내기 (CSV 다운로드 및 드라이브 저장)

In [ ]:
# 1. 코랩 인스턴스에 CSV 저장
output_path = '/content/category_summary.csv'
category_summary.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"💾 파일이 코랩 스토리지({output_path})에 저장되었습니다.")

# 2. 브라우저로 로컬 PC 다운로드 (필요시 주석 해제)
# from google.colab import files
# files.download(output_path)

# 3. 구글 드라이브에 저장 (드라이브 마운트 후 주석 해제)
# category_summary.to_csv('/content/drive/MyDrive/category_summary.csv', index=False, encoding='utf-8-sig')